# 🎨 Interfaz Web con Streamlit

Aplicación interactiva para predecir precios de propiedades en Dinamarca usando los 4 modelos entrenados:
- **XGBoost** (CPU optimizado)
- **RandomForest** (Mejor RMSE: 38,812 DKK)
- **Ridge** (GLM con regularización L2)
- **FLAML AutoML** (Optimización automática)

## 1. Contrato de entrada (lo que pide el endpoint/formulario)
- `date` (YYYY-MM-DD) o `year` si no hay fecha exacta.
- `region`
- `house_type`
- `sales_type`
- `sqm` (metros cuadrados)
- `no_rooms`
- `year_build`

> Con `region` el backend puede generar las features geográficas sintéticas de `src/features/geospatial_features.py`.

## 2. Features que el backend puede derivar (sin conocer el precio)
- **Temporales:** `year`, `month`, `quarter`, `season`, `month_sin`, `month_cos`, `quarter_sin`, `quarter_cos`, `crisis_period`, `market_phase`, `time_trend`.
- **Tamaño/espacio:** `sqm_per_room`, `rooms_sqm_ratio`, `rooms_category`, `size_category`.
- **Categóricas codificadas:** one-hot de `house_type`, `sales_type`, `season`; `region_frequency`.
- **Geográficas sintéticas:** `urban_density`, `distance_to_center`, `location_type`, `transport_access`, `geo_cluster` (se calculan en backend con `region`).

## 🚀 Instalación y Ejecución

### 1. Instalar Streamlit
```bash
pip install streamlit plotly
```

### 2. Ejecutar la aplicación
```bash
streamlit run app.py
```

La aplicación se abrirá automáticamente en tu navegador en `http://localhost:8501`

---

## 🎯 Características de la Aplicación

### Interfaz Interactiva
- **Sidebar** con formulario para ingresar datos de la propiedad
- **Predicciones en tiempo real** de los 4 modelos
- **Visualizaciones comparativas** con gráficos Plotly
- **Diseño responsive** con CSS personalizado

### Inputs del Usuario (Sidebar)
1. **Fecha de venta** (date picker)
2. **Región** (dropdown): Capital Region, Zealand, Southern Denmark, Central Jutland, North Jutland
3. **Tipo de propiedad** (dropdown): Apartment, Terraced house, Villa, Semi-detached house
4. **Tipo de venta** (dropdown): Regular Sale, Foreclosure, Other
5. **Área en m²** (number input): 10-1000
6. **Número de habitaciones** (number input): 1-20
7. **Año de construcción** (number input): 1800-2024

### Outputs
- **Precio promedio estimado** (grande y destacado)
- **Predicciones individuales** por modelo (4 tarjetas)
- **Gráfico comparativo** de barras
- **Estadísticas** (min, max, desviación estándar)
- **Features generadas** (tabla técnica)
- **Métricas de rendimiento** de los modelos

---

## 🔧 Arquitectura Técnica

### Feature Engineering Automático
La aplicación genera automáticamente **24 features** a partir de los 7 inputs del usuario:

#### Features Temporales
- `year`, `month_sin`, `month_cos`, `quarter_sin`, `quarter_cos`

#### Features de Tamaño y Espacio
- `sqm`, `no_rooms`, `rooms_sqm_ratio`, `price_per_sqm`

#### Features de Edad
- `property_age = año_actual - año_construcción`
- `age_x_villa` (interacción)

#### Features Geográficas
- `region_frequency`, `region_price_mean`, `sqm_x_region`, `price_per_sqm_x_region`

#### Features Categóricas (One-Hot)
- `is_premium` (1 si es Villa)
- `sales_type_regular_sale`
- `price_category_Premium`, `price_category_High`, `price_category_Medium`

### Pipeline de Predicción
1. **Input del usuario** → 7 campos del formulario
2. **Feature Engineering** → Genera 24 features numéricas
3. **Scaling** → StandardScaler ajustado con datos de entrenamiento
4. **Predicción** → 4 modelos predicen `log_price` en paralelo
5. **Transformación** → `exp(log_price)` para obtener precio en DKK
6. **Visualización** → Comparación y estadísticas

---